# ♟️ Modular Chess Engine V1.5 — Обучение нейросети на 2.3 млн позиций с GPU T4
Этот блокнот обучает **усиленную версию шахматного ИИ (ModularChessNet-V1.5)** методом дистилляции знаний от гроссмейстерского движка **Stockfish**.

### 🚀 Ключевые отличия V1.5 от V1.0:
* **Объём данных**: **2 300 000 позиций** (вместо 1.5 млн в V1) из базы Lichess Stockfish Evaluations.
* **Ёмкость модели**: **6 580 000 параметров** (~6.58M, почти в 3 раза больше, чем 2.3M у V1).
* **Ширина и глубина**: **8 остаточных блоков** (ResBlocks) и **192 канала** (вместо 6 блоков и 128 каналов).
* **Внимание к полям (Channel Attention)**: Добавлены слои **Squeeze-and-Excitation (SE-ResNet)** — модель динамически фокусируется на угрозах королю, связках и вилках.
* **Аппаратное ускорение**: Полная утилизация тензорных ядер **NVIDIA Tesla T4 (FP16 AMP)**.
* **Экономия RAM**: Векторизация в формате `uint8` с преобразованием во float на GPU (гарантия от вылетов по ОЗУ в Colab).
* **Экспорт**: Готовые файлы весов `modular_chess_v1_5.pt` и `modular_chess_v1_5.onnx` для локального сервера и браузера.

> **💡 Перед запуском**: Убедитесь, что включен T4 GPU:  
> **Среда выполнения (Runtime)** ➔ **Сменить среду (Change runtime type)** ➔ **T4 GPU** ➔ **Сохранить**.

In [ ]:
# 1. Проверка GPU и тензорных ядер (Tensor Cores)
import torch

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemError("⚠️ GPU не найден! Включите T4 GPU в меню: Среда выполнения -> Сменить среду -> T4 GPU.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)

print(f"✅ GPU: {gpu_name}")
print(f"⚡ Compute Capability: {capability[0]}.{capability[1]}")

if capability[0] >= 7:
    print("🔥 Тензорные ядра Turing (Tensor Cores) активны! Режим смешанной точности FP16 включен.")
else:
    print("ℹ️ Базовый режим CUDA.")

# Автонастройка сверток cuDNN
torch.backends.cudnn.benchmark = True

In [ ]:
# 2. Установка зависимостей
!pip install -q python-chess pyarrow datasets tqdm onnx matplotlib
print("✅ Зависимости успешно установлены!")

In [ ]:
# 3. Загрузка обучающих данных Stockfish (Part 0 и Part 1)
import os
import urllib.request

dataset_files = [
    ("stockfish_evals_part0.parquet", "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"),
    ("stockfish_evals_part1.parquet", "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/refs%2Fconvert%2Fparquet/default/train/0001.parquet")
]

for fname, url in dataset_files:
    if not os.path.exists(fname):
        print(f"📥 Скачивание {fname} с HuggingFace (скорость сети Colab ~150-250 МБ/с)...")
        urllib.request.urlretrieve(url, fname)
        print(f"✅ {fname} успешно загружен!")
    else:
        print(f"ℹ️ {fname} уже найден на диске.")

In [ ]:
# 4. Векторизация 2 300 000 позиций (Оптимизация uint8 для Colab RAM)
import pyarrow.parquet as pq
import numpy as np
from tqdm.auto import tqdm

TARGET_SAMPLES = 2_300_000

PIECE_MAP = {
    'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
    'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11
}

def fen_to_tensor_fast(fen: str) -> np.ndarray:
    planes = np.zeros((16, 8, 8), dtype=np.uint8)
    parts = fen.split()
    board_str = parts[0]
    turn_str = parts[1] if len(parts) > 1 else 'w'
    castling_str = parts[2] if len(parts) > 2 else '-'
    
    r, c = 0, 0
    for ch in board_str:
        if ch == '/':
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            p_idx = PIECE_MAP.get(ch)
            if p_idx is not None:
                planes[p_idx, r, c] = 1
            c += 1
            
    if turn_str == 'w':
        planes[12, :, :] = 1
    if 'K' in castling_str: planes[13, 0:4, 0:4] = 1
    if 'Q' in castling_str: planes[13, 0:4, 4:8] = 1
    if 'k' in castling_str: planes[14, 4:8, 0:4] = 1
    if 'q' in castling_str: planes[14, 4:8, 4:8] = 1
    return planes

def uci_to_policy_idx(uci: str):
    if not uci or len(uci) < 4:
        return None
    from_c = ord(uci[0]) - 97
    from_r = 8 - int(uci[1])
    to_c = ord(uci[2]) - 97
    to_r = 8 - int(uci[3])
    if 0 <= from_r < 8 and 0 <= from_c < 8 and 0 <= to_r < 8 and 0 <= to_c < 8:
        return (from_r * 8 + from_c) * 64 + (to_r * 8 + to_c)
    return None

print(f"⚙️ Извлечение и кодирование {TARGET_SAMPLES:,} позиций Stockfish...")
all_boards = np.zeros((TARGET_SAMPLES, 16, 8, 8), dtype=np.uint8)
all_policies = np.zeros(TARGET_SAMPLES, dtype=np.int64)
all_values = np.zeros((TARGET_SAMPLES, 1), dtype=np.float32)

loaded = 0
pbar = tqdm(total=TARGET_SAMPLES, desc="Обработка позиций")

for fname, _ in dataset_files:
    if loaded >= TARGET_SAMPLES:
        break
    parquet_file = pq.ParquetFile(fname)
    for batch in parquet_file.iter_batches(batch_size=32768, columns=['fen', 'line', 'cp', 'mate']):
        df = batch.to_pandas()
        for _, row in df.iterrows():
            line = row['line']
            if not line:
                continue
            first_move = line.split()[0]
            pol_idx = uci_to_policy_idx(first_move)
            if pol_idx is None:
                continue
            cp = row['cp']
            mate = row['mate']
            if mate is not None and not np.isnan(mate):
                val = 1.0 if mate > 0 else -1.0
            elif cp is not None and not np.isnan(cp):
                val = float(np.tanh(cp / 400.0))
            else:
                continue
            all_boards[loaded] = fen_to_tensor_fast(row['fen'])
            all_policies[loaded] = pol_idx
            all_values[loaded, 0] = val
            loaded += 1
            pbar.update(1)
            if loaded >= TARGET_SAMPLES:
                break
        if loaded >= TARGET_SAMPLES:
            break
pbar.close()
print(f"🎉 Успешно подготовлено {loaded:,} позиций в памяти!")

In [ ]:
# 5. Создание DataLoader с Pinned Memory
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.from_numpy(all_boards[:loaded])
y_policy = torch.from_numpy(all_policies[:loaded])
y_value = torch.from_numpy(all_values[:loaded])

del all_boards, all_policies, all_values

dataset = TensorDataset(X_tensor, y_policy, y_value)
BATCH_SIZE = 1024  # Кратность 128 для Tensor Cores

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

print(f"📦 Готово {len(train_loader)} батчей на эпоху (Batch size: {BATCH_SIZE})")

In [ ]:
# 6. Архитектура нейросети ModularChessNet-V1.5 (8 SE-ResBlocks, 192 Channels)
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channels=192, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean((2, 3))
        y = self.relu(self.fc1(y))
        y = self.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class ResBlockSE(nn.Module):
    def __init__(self, channels=192):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction=4)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return self.relu(out + res)

class ModularChessNetV15(nn.Module):
    def __init__(self, num_res_blocks=8, channels=192):
        super().__init__()
        self.in_conv = nn.Conv2d(16, channels, kernel_size=3, padding=1, bias=False)
        self.in_bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

        self.blocks = nn.ModuleList([ResBlockSE(channels) for _ in range(num_res_blocks)])

        # Policy Head (4 канала 8x8 = 256 -> 4096 классов ходов)
        self.policy_conv = nn.Conv2d(channels, 4, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(4)
        self.policy_fc = nn.Linear(4 * 8 * 8, 4096)

        # Value Head (2 канала 8x8 = 128 -> 256 -> 1)
        self.value_conv = nn.Conv2d(channels, 2, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(2)
        self.value_fc1 = nn.Linear(2 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        out = self.relu(self.in_bn(self.in_conv(x)))
        for block in self.blocks:
            out = block(out)

        p = self.relu(self.policy_bn(self.policy_conv(out)))
        policy_logits = self.policy_fc(p.flatten(1))

        v = self.relu(self.value_bn(self.value_conv(out)))
        v = self.relu(self.value_fc1(v.flatten(1)))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value

model = ModularChessNetV15(num_res_blocks=8, channels=192).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🧠 Создана модель ModularChessNet-V1.5!")
print(f"📊 Параметров: {param_count:,} (~{param_count/1e6:.2f}M, прирост в 2.8x по сравнению с V1)")

In [ ]:
# 7. Запуск обучения (15 эпох) с FP16 Tensor Cores
import time
import matplotlib.pyplot as plt

EPOCHS = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

policy_criterion = nn.CrossEntropyLoss()
value_criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

history = {'loss': [], 'policy_loss': [], 'value_loss': []}

print(f"🚀 Старт обучения V1.5 на 2.3M позиций (GPU: {gpu_name})...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss_epoch = 0.0
    total_p_loss = 0.0
    total_v_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch}/{EPOCHS}", leave=False)
    
    for boards_uint8, targets_p, targets_v in pbar:
        # Быстрая конвертация uint8 -> float32 на GPU
        boards = boards_uint8.to(device, non_blocking=True).to(torch.float32)
        targets_p = targets_p.to(device, non_blocking=True)
        targets_v = targets_v.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            pred_p, pred_v = model(boards)
            p_loss = policy_criterion(pred_p, targets_p)
            v_loss = value_criterion(pred_v, targets_v)
            loss = p_loss + 0.5 * v_loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss_epoch += loss.item()
        total_p_loss += p_loss.item()
        total_v_loss += v_loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'p': f"{p_loss.item():.4f}", 'v': f"{v_loss.item():.4f}"})
        
    scheduler.step()
    
    avg_loss = total_loss_epoch / len(train_loader)
    avg_p = total_p_loss / len(train_loader)
    avg_v = total_v_loss / len(train_loader)
    
    history['loss'].append(avg_loss)
    history['policy_loss'].append(avg_p)
    history['value_loss'].append(avg_v)
    
    print(f"📊 Эпоха {epoch:02d}/{EPOCHS} завершена | Loss: {avg_loss:.4f} (Policy: {avg_p:.4f}, Value: {avg_v:.4f}) | LR: {scheduler.get_last_lr()[0]:.6f}")

total_min = (time.time() - start_time) / 60
print(f"🎉 Обучение V1.5 завершено за {total_min:.2f} минут!")

plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS+1), history['loss'], label='Total Loss', color='blue', lw=2)
plt.plot(range(1, EPOCHS+1), history['policy_loss'], label='Policy Loss', color='orange', linestyle='--')
plt.plot(range(1, EPOCHS+1), history['value_loss'], label='Value Loss', color='green', linestyle=':')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Кривая обучения ModularChessNet-V1.5 (2.3 млн позиций)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 8. Интерактивная проверка ходов V1.5 прямо в блокноте
import chess
from IPython.display import display

def select_best_move(board: chess.Board, model, device):
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None, 0.0
        
    fen_tensor = fen_to_tensor_fast(board.fen())
    inp = torch.from_numpy(fen_tensor).unsqueeze(0).to(device).to(torch.float32)
    
    model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.float16):
            policy_logits, value = model(inp)
            policy_probs = torch.softmax(policy_logits, dim=-1).squeeze(0).cpu().numpy()
            
    best_move = None
    best_score = -1e9
    
    for m in legal_moves:
        idx = uci_to_policy_idx(m.uci())
        score = policy_probs[idx] if idx is not None else 0.0
        if score > best_score:
            best_score = score
            best_move = m
            
    return best_move, float(value.squeeze().cpu().numpy())

test_board = chess.Board()
print("Тест начальной позиции:")
display(test_board)

move, pos_eval = select_best_move(test_board, model, device)
print(f"🤖 Оценка V1.5: {pos_eval:+.3f} (от -1.0 до +1.0)")
print(f"🤖 Выбранный ход: {move.uci()} ({test_board.san(move)})")
test_board.push(move)
display(test_board)

In [ ]:
# 9. Экспорт модели в PyTorch (.pt) и ONNX (.onnx) с автоскачиванием
model.eval()

PT_NAME = "modular_chess_v1_5.pt"
ONNX_NAME = "modular_chess_v1_5.onnx"

torch.save(model.state_dict(), PT_NAME)
print(f"✅ Веса сохранены: {PT_NAME}")

dummy_input = torch.randn(1, 16, 8, 8, device=device)
torch.onnx.export(
    model,
    dummy_input,
    ONNX_NAME,
    input_names=["board_state"],
    output_names=["policy_logits", "value"],
    dynamic_axes={
        "board_state": {0: "batch_size"},
        "policy_logits": {0: "batch_size"},
        "value": {0: "batch_size"}
    },
    opset_version=14
)
print(f"✅ ONNX модель экспортирована: {ONNX_NAME}")

# Автоматическое скачивание файлов на компьютер
try:
    from google.colab import files
    print("📥 Загрузка файлов на ваш компьютер...")
    files.download(ONNX_NAME)
    files.download(PT_NAME)
except Exception as e:
    print(f"ℹ️ Файлы сохранены в проводнике Google Colab: {ONNX_NAME}, {PT_NAME}")